<a href="https://colab.research.google.com/github/santoromarco74/adv_comp_proj/blob/main/adv_comp_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Box blur 3×3 su GPU — CPU vs memoria globale vs shared memory

Progetto per **Advanced Computer Architecture**, traccia *Filters on images*.

Il filtro sostituisce ogni pixel con la media dei suoi 9 vicini. Lo stesso calcolo
è implementato tre volte e confrontato:

| Versione | Come funziona |
|---|---|
| **CPU sequenziale** | due cicli annidati, un pixel alla volta, un solo core |
| **GPU naive** | un thread per pixel, i 9 vicini letti dalla memoria globale |
| **GPU shared memory** | il blocco copia un tile 18×18 (16×16 + halo) in shared memory, poi calcola da lì |

**Risultato principale:** su Tesla T4 la versione con shared memory è **più lenta**
di quella naive di un fattore 1,35–1,53. In uno stencil 3×3 il riuso è solo 9× e
l'area di lavoro è 324 byte: la cache L1 fa già il tiling da sola, quindi la shared
memory paga barriera, divergenza dei rami e sbilanciamento del carico senza
incassare alcun beneficio.

> **Nota sul metodo.** Ogni tempo è misurato **dopo un warm-up** e mediato su 100
> ripetizioni. Senza warm-up il primo kernel lanciato assorbe il costo una tantum
> di caricamento del modulo e compilazione JIT, e risulta artificialmente
> lentissimo. È il motivo per cui una versione precedente di questo notebook
> concludeva, sbagliando, che la GPU fosse più lenta della CPU.

## 1. La GPU a disposizione

In [1]:
!nvidia-smi

Fri Aug 28 14:52:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. I sorgenti

Tre file, scaricati dal repository:

- **`benchmark.cu`** — i due kernel, la reference CPU e tutta la misura
- **`test_host.cu`** — verifica di correttezza eseguibile anche senza GPU
- **`real_image_blur.cu`** — applica il filtro a una fotografia vera

`nvcc` va sempre invocato con **`-arch=sm_75`** (l'architettura della T4): senza,
la scheda deve ricompilare il PTX al primo lancio e i tempi risultano falsati.

In [ ]:
BRANCH = "claude/advanced-architecture-project-bn1uoh"   # dopo il merge diventa "main"
RAW = f"https://raw.githubusercontent.com/santoromarco74/adv_comp_proj/{BRANCH}"

for f in ["benchmark.cu", "test_host.cu", "real_image_blur.cu"]:
    !wget -q -O {f} {RAW}/{f}

!ls -l benchmark.cu test_host.cu real_image_blur.cu

## 3. Verifica di correttezza

`test_host.cu` emula i due kernel sulla CPU replicandone esattamente
l'indicizzazione, e controlla che diano lo stesso risultato della reference —
anche su immagini di lato **non multiplo di 16**, dove l'ultimo blocco di thread
sborda (100×37, 17×17, 3×200, 1×1). Verifica inoltre che nessun thread legga una
cella di shared memory mai inizializzata.

In [27]:
!nvcc -O2 -arch=sm_75 test_host.cu -o test_host && ./test_host

=== fillImage: determinismo e distribuzione ===
  ok  : stesso seme -> stessa immagine
  ok  : semi diversi -> immagini diverse
  ok  : usa molti livelli di grigio (256/256)

=== computeStats ===
  ok  : media = 5
  ok  : deviazione standard campionaria = 2.1381
  ok  : mediana = 4.5
  ok  : minimo = 2
  ok  : campione singolo non divide per zero

=== parseSizes ===
  ok  : lista valida
  ok  : valore singolo
  ok  : rifiuta campo vuoto
  ok  : rifiuta valore negativo
  ok  : rifiuta stringa vuota

=== equivalenza dei kernel (emulati sull'host) ===

--- immagine 64x64 ---
  ok  : naive  == reference CPU  (0 differenze)
  ok  : shared == reference CPU  (0 differenze)
  ok  : nessuna lettura di shared memory non inizializzata (0)

--- immagine 128x96 ---
  ok  : naive  == reference CPU  (0 differenze)
  ok  : shared == reference CPU  (0 differenze)
  ok  : nessuna lettura di shared memory non inizializzata (0)

--- immagine 100x37 ---
  ok  : naive  == reference CPU  (0 differenze)
  ok 

## 4. Benchmark

Per ogni risoluzione: 10 lanci di riscaldamento, poi 100 ripetizioni cronometrate
con `cudaEvent`, riportate come media ± deviazione standard. I trasferimenti
host↔device sono misurati a parte, e lo speed-up è dato sia sul solo kernel sia
end-to-end (copia andata + kernel + copia ritorno).

`--order-check` rimisura i due kernel a ordine invertito: se restasse un effetto
di posizione, il kernel che passa da primo a secondo migliorerebbe vistosamente
mentre l'altro peggiora.

> Il valore della CPU è la misura più rumorosa della tabella (poche ripetizioni su
> una macchina condivisa). Per il report conviene rilanciare con **`--cpu-reps 10`**.

In [26]:
!nvcc -O3 -arch=sm_75 benchmark.cu -o benchmark
!./benchmark --csv risultati.csv --order-check

GPU ............ Tesla T4 (compute capability 7.5)
Multiprocessori  40
Shared memory .. 48 KB per blocco, 64 KB per multiprocessore
Banda di picco .. 320.1 GB/s
Configurazione .. blocco 16x16, tile in shared memory 18x18
Misura ......... 100 ripetizioni dopo 10 lanci di riscaldamento, CPU 3 ripetizioni, seme 42

[512x512] in corso...
  controllo ordine  naive 0.0180 -> 0.0178 ms (-1.2%)   shared 0.0246 -> 0.0245 ms (-0.4%)

[1024x1024] in corso...
  controllo ordine  naive 0.0522 -> 0.0554 ms (+6.1%)   shared 0.0783 -> 0.0836 ms (+6.7%)

[2048x2048] in corso...
  controllo ordine  naive 0.2381 -> 0.1769 ms (-25.7%)   shared 0.3641 -> 0.2580 ms (-29.1%)

[4096x4096] in corso...
  controllo ordine  naive 0.6304 -> 0.6173 ms (-2.1%)   shared 0.8540 -> 0.7878 ms (-7.7%)

Tempi in millisecondi: media +/- deviazione standard
Risoluzione                CPU         copia H->D          GPU naive         GPU shared         copia D->H
--------------------------------------------------------------

## 5. Risultati

I grafici sono generati **dal CSV appena prodotto**, non da valori scritti a mano:
se rilanci il benchmark, si aggiornano da soli.

In [28]:
import pandas as pd
df = pd.read_csv('risultati.csv')
df

,dim,cpu_ms,cpu_sd,h2d_ms,h2d_sd,naive_ms,naive_sd,shared_ms,shared_sd,d2h_ms,d2h_sd,speedup_naive_kernel,speedup_shared_kernel,speedup_naive_e2e,speedup_shared_e2e,shared_vs_naive,verify_naive,verify_shared
0,512,4.448652,0.060501,0.074220,0.001217,0.017976,0.000730,0.024635,0.001771,0.084810,0.004904,247.4817,180.5835,25.1328,24.2215,0.7297,1,1
1,1024,14.959653,0.545235,0.335735,0.154405,0.052245,0.001344,0.078273,0.001540,0.386724,0.173202,286.3341,191.1216,19.3101,18.6825,0.6675,1,1
2,2048,46.233337,16.836399,0.910844,0.031322,0.238064,0.001043,0.364086,0.000578,0.978703,0.023838,194.2052,126.9847,21.7302,20.5150,0.6539,1,1
3,4096,151.977061,5.622215,3.438220,0.193498,0.630439,0.040441,0.853987,0.001179,3.683916,0.130447,241.0656,177.9618,19.6034,19.0540,0.7382,1,1


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.transforms import blended_transform_factory

df = pd.read_csv("risultati.csv")

# Palette categorica validata per daltonismo e contrasto.
SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dcdcd8"
C_CPU, C_NAIVE, C_SHARED = "#2a78d6", "#eb6834", "#1baf7a"

labels = [f"{d}$^2$" for d in df["dim"]]
x = np.arange(len(df))

def style(ax, ylabel, title):
    ax.set_facecolor(SURF)
    ax.set_ylabel(ylabel, fontsize=10, color=INK2)
    ax.set_title(title, fontsize=12, fontweight="bold", color=INK, pad=12, loc="left")
    ax.set_xticks(x); ax.set_xticklabels(labels, color=INK2)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

def label_bars(ax, xs, vals, fmt="{:.3g}", tops=None):
    """Etichetta ogni barra. `tops` permette di scavalcare la barra d'errore."""
    tops = vals if tops is None else tops
    for xi, v, t in zip(xs, vals, tops):
        ax.annotate(fmt.format(v), (xi, t), ha="center", va="bottom",
                    fontsize=8, color=INK2, xytext=(0, 4), textcoords="offset points")

# --- 1. tempi di esecuzione -------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.8), facecolor=SURF)
w = 0.27
for off, col, sd, c, lab in [
        (-w - 0.012, "cpu_ms",    "cpu_sd",    C_CPU,    "CPU sequenziale"),
        (0,          "naive_ms",  "naive_sd",  C_NAIVE,  "GPU naive (memoria globale)"),
        (w + 0.012,  "shared_ms", "shared_sd", C_SHARED, "GPU shared memory")]:
    ax.bar(x + off, df[col], w, label=lab, color=c,
           yerr=df[sd], error_kw=dict(ecolor=INK2, lw=1, capsize=3))
    label_bars(ax, x + off, df[col], tops=df[col] + df[sd])
ax.set_yscale("log")
style(ax, "Tempo (ms, scala logaritmica)", "Tempo di esecuzione")
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper left")
fig.tight_layout(); fig.savefig("fig1_tempi.png", dpi=200, facecolor=SURF); plt.show()

# --- 2. il risultato principale --------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor=SURF)
ratio = df["shared_ms"] / df["naive_ms"]
ax.bar(x, ratio, 0.5, color=C_SHARED)
label_bars(ax, x, ratio, "{:.2f}x")
ax.axhline(1.0, color=C_NAIVE, lw=2, zorder=3)
ax.annotate("pareggio: sopra questa linea la shared memory costa di più",
            (0.995, 1.0), xycoords=blended_transform_factory(ax.transAxes, ax.transData),
            xytext=(0, 6), textcoords="offset points",
            ha="right", va="bottom", fontsize=9, color=C_NAIVE, fontweight="bold")
ax.set_ylim(0, max(ratio) * 1.35)
style(ax, "tempo shared / tempo naive", "La shared memory è più lenta a ogni risoluzione")
fig.tight_layout(); fig.savefig("fig2_shared_vs_naive.png", dpi=200, facecolor=SURF); plt.show()

# --- 3. dove finisce il tempo ----------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.8), facecolor=SURF)
bottom = np.zeros(len(df))
for col, c, lab in [("h2d_ms", C_CPU, "copia RAM → GPU"),
                    ("naive_ms", C_NAIVE, "kernel naive"),
                    ("d2h_ms", C_SHARED, "copia GPU → RAM")]:
    ax.bar(x, df[col], 0.5, bottom=bottom, label=lab, color=c, linewidth=2, edgecolor=SURF)
    bottom += df[col]
for xi, tot, k in zip(x, bottom, df["naive_ms"]):
    ax.annotate(f"{100 * (tot - k) / tot:.0f}% in trasferimenti", (xi, tot),
                ha="center", va="bottom", fontsize=8.5, color=INK2,
                xytext=(0, 4), textcoords="offset points")
ax.set_ylim(0, max(bottom) * 1.18)
style(ax, "Tempo (ms)", "Il collo di bottiglia non è il calcolo, è il bus PCIe")
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper left")
fig.tight_layout(); fig.savefig("fig3_ripartizione.png", dpi=200, facecolor=SURF); plt.show()

# --- 4. banda effettiva contro il picco ------------------------------------
PEAK = 320.1   # GB/s, Tesla T4
moved = 2 * df["dim"].astype(float) ** 2 / 1e9        # letture + scritture
fig, ax = plt.subplots(figsize=(9, 4.4), facecolor=SURF)
for off, col, c, lab in [(-0.14, "naive_ms", C_NAIVE, "GPU naive"),
                         (0.14, "shared_ms", C_SHARED, "GPU shared memory")]:
    bw = moved / (df[col] / 1000.0)
    ax.bar(x + off, bw, 0.26, label=lab, color=c)
    label_bars(ax, x + off, bw, "{:.0f}")
ax.axhline(PEAK, color=INK2, lw=1.5, ls="--")
ax.annotate(f"picco teorico della T4: {PEAK:.0f} GB/s", (0.995, PEAK),
            xycoords=blended_transform_factory(ax.transAxes, ax.transData),
            xytext=(0, 5), textcoords="offset points", ha="right", va="bottom",
            fontsize=9, color=INK2)
ax.set_ylim(0, PEAK * 1.15)
style(ax, "Banda effettiva (GB/s)", "Nessun kernel supera il 17% della banda disponibile")
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper left")
fig.tight_layout(); fig.savefig("fig4_banda.png", dpi=200, facecolor=SURF); plt.show()

## 6. Profiling

`nvprof` legge la timeline della GPU, quindi misura il tempo dei kernel
indipendentemente da come il programma li cronometra: è la conferma esterna del
risultato. Qui gira su una singola risoluzione con poche ripetizioni, perché il
profiler aggiunge overhead a ogni lancio.

> `nvprof` è deprecato e la T4 è l'ultima architettura che supporta. Su schede più
> recenti, e per metriche come *achieved occupancy* ed efficienza degli accessi
> globali, si usa Nsight Compute: `ncu --set full ./benchmark --sizes 2048 --reps 5`.

In [ ]:
!nvprof ./benchmark --sizes 2048 --reps 20 --warmup 5 --cpu-reps 1

## 7. Prova su una fotografia vera

Fin qui le immagini erano sintetiche. Questa sezione applica il filtro a un JPG
reale e ne salva il risultato, verificando che la GPU produca esattamente gli
stessi pixel della CPU.

`real_image_blur.cu` fa `#include "benchmark.cu"` e riusa gli stessi kernel,
invece di ricopiarli.

In [ ]:
# Librerie header-only per leggere e scrivere immagini
!wget -q https://raw.githubusercontent.com/nothings/stb/master/stb_image.h
!wget -q https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h

# Immagine di prova. L'URL ha un seed fisso: scarica sempre la stessa foto,
# cosi' la prova e' riproducibile.
!wget -q -O input.jpg https://picsum.photos/seed/aca2026/2048/2048
!ls -l input.jpg

In [ ]:
!nvcc -O3 -arch=sm_75 real_image_blur.cu -o real_image_blur 2>/dev/null
!./real_image_blur input.jpg output_blur.png

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(12, 6), facecolor="#fcfcfb")
for ax, path, title in [(axes[0], "input.jpg", "Originale (scala di grigi)"),
                        (axes[1], "output_blur.png", "Filtrata su GPU")]:
    img = Image.open(path)
    ax.imshow(img.convert("L"), cmap="gray")
    ax.set_title(title, fontweight="bold", fontsize=11, color="#0b0b0b")
    ax.axis("off")
fig.tight_layout()
fig.savefig("confronto_visivo.png", dpi=200, facecolor="#fcfcfb")
plt.show()